# PyGeoFetch — 04: Download & Post-Processing

Covers the full download API: band selection, parallel downloads, BUG 2/3/4 fixes,
and the full inline post-processing chain.

### What you'll learn
- BUG 2: len(results) always equals len(items)
- BUG 3: Partial downloads detected via _validate_downloaded_file()
- BUG 4: Reprojection identity transform detection and rejection
- Band selection (reduce 600 MB to 150 MB)
- 9 inline post-processing actions: reproject, COG, clip, compress, ndvi, merge…
- Inspect downloaded files with rasterio

In [3]:
from pathlib import Path

from pygeofetch import PyGeoFetch
from pygeofetch.models.download_task import DownloadOptions, PostProcessAction
from pygeofetch.models.search_query import BoundingBox, SearchQuery

client = PyGeoFetch(log_level="INFO")
Path("output").mkdir(exist_ok=True)
nyc_bbox = BoundingBox.from_string("-74.1,40.6,-73.7,40.9")

# Search for clear scenes to download
query = SearchQuery(
    bbox=nyc_bbox,
    start_date="2024-01-01",
    end_date="2024-06-01",
    cloud_cover_max=10,
    max_results=10,
    sort_by="cloud_cover",
    sort_ascending=True,
)
results = client.search(query, providers=["aws_earth"])
print(f"Found {len(results)} scenes to download from")

23:00:49 INFO [      engine] PyGeoFetch ready
Found 10 scenes to download from


## 1. BUG 2 — download() Length/Order Contract

In [4]:
# BUG 2 fix: len(results) ALWAYS equals len(items), even when some fail.
# Results are in the same order as input items.
# Failed items return DownloadResult(success=False, error=...) not None or omitted.

from pygeofetch.core.downloader import AdaptiveDownloader

# Demonstrate the contract
dm = AdaptiveDownloader()
items = results[:1] if len(results) >= 3 else results
options = DownloadOptions(parallel=1, on_failure="skip")

print("Contract verification:")
print(f"  Input  items  : {len(items)}")

# Real download (or simulate)
dl_results = client.download(items, Path("output/downloads/"), options)

print(f"  Output results: {len(dl_results)}")
print(f"  len(results) == len(items): {len(dl_results) == len(items)}")
print()

for i, (item, result) in enumerate(zip(items, dl_results)):
    id_match = result.data_id == item.id
    print(f"  [{i}] data_id matches: {id_match}  success={result.success}")
    if not result.success:
        print(f"       error: {result.error[:80]}")

Contract verification:
  Input  items  : 1


23:00:55 INFO [  downloader] Downloading S2B_18TXK_20240531_0_L2A
23:00:55 INFO [  downloader]   ✓ S2B_18TXK_20240531_0_L2A                        1064 MB    0.0s
  Output results: 1
  len(results) == len(items): True

  [0] data_id matches: True  success=True


## 2. BUG 3 — File Validation After Download

In [17]:
# BUG 3 fix: _validate_downloaded_file() reads the first tile of every raster.
# A partial download that GDAL can open (but has truncated pixels) is now detected.

import tempfile

from pygeofetch.core.downloader import AdaptiveDownloader

dm = AdaptiveDownloader()

# Create test rasters to demonstrate validation
try:
    import numpy as np
    import rasterio
    from rasterio.crs import CRS
    from rasterio.transform import from_bounds

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)

        # 1. Valid GeoTIFF
        valid_path = tmp / "valid.tif"
        with rasterio.open(valid_path, "w",
                           driver="GTiff", dtype="uint16", count=1, width=20, height=20,
                           crs=CRS.from_epsg(4326),
                           transform=from_bounds(-74.1, 40.6, -73.7, 40.9, 20, 20)) as ds:
            ds.write(np.ones((1, 20, 20), dtype=np.uint16))
        is_valid, err = dm._validate_downloaded_file(valid_path)
        print(f"Valid GeoTIFF:    is_valid={is_valid} (expected True)   err={err!r}")

        # 2. Empty file (0 bytes) — simulates interrupted download
        empty_path = tmp / "empty.tif"
        empty_path.write_bytes(b"")
        is_valid, err = dm._validate_downloaded_file(empty_path)
        print(f"Empty file:       is_valid={is_valid} (expected False)  err={err!r}")

        # 3. Non-raster: JSON metadata
        json_path = tmp / "metadata.json"
        json_path.write_text('{"scene_id": "S1C_IW_20260601"}')
        is_valid, err = dm._validate_downloaded_file(json_path)
        print(f"JSON metadata:    is_valid={is_valid} (expected True)   err={err!r}")

        # 4. Empty JSON — would fail
        empty_json = tmp / "empty.json"
        empty_json.write_bytes(b"")
        is_valid, err = dm._validate_downloaded_file(empty_json)
        print(f"Empty JSON:       is_valid={is_valid} (expected False)  err={err!r}")

except ImportError:
    print("rasterio not installed — pip install 'PyGeoFetch[geo]'")

Valid GeoTIFF:    is_valid=True (expected True)   err=''
Empty file:       is_valid=False (expected False)  err='File is empty (0 bytes)'
JSON metadata:    is_valid=True (expected True)   err=''
Empty JSON:       is_valid=False (expected False)  err='File is empty (0 bytes)'


## 3. BUG 4 — CRS Identity Transform Detection

In [18]:
# BUG 4 fix: After reproject:EPSG:XXXX, some scenes ended up with
# pixel_width=1.0, origin=(0.0, 0.0) in a metre-unit CRS.
# _has_identity_transform() detects this.
# _reproject_with_validation() raises RuntimeError if it occurs.

try:
    import tempfile

    import numpy as np
    import rasterio
    from rasterio.crs import CRS
    from rasterio.transform import Affine, from_bounds

    from pygeofetch.core.downloader import AdaptiveDownloader

    dm = AdaptiveDownloader()

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)

        # Buggy file: identity transform in projected CRS
        identity_path = tmp / "identity.tif"
        with rasterio.open(identity_path, "w",
                           driver="GTiff", dtype="uint16", count=1, width=10, height=10,
                           crs=CRS.from_epsg(32632), transform=Affine(1,0,0, 0,-1,0)) as ds:
            ds.write(np.ones((1,10,10), dtype=np.uint16))

        print(f"Identity transform detected: {dm._has_identity_transform(identity_path)}")
        print("  (pixel=1.0m, origin=(0,0), projected CRS — geographic coordinates wrong)")

        # Good file: real WGS84 transform
        good_path = tmp / "good.tif"
        with rasterio.open(good_path, "w",
                           driver="GTiff", dtype="uint16", count=1, width=10, height=10,
                           crs=CRS.from_epsg(4326),
                           transform=from_bounds(-74.1, 40.6, -73.7, 40.9, 10, 10)) as ds:
            ds.write(np.ones((1,10,10), dtype=np.uint16))

        print(f"Normal WGS84 file detected: {dm._has_identity_transform(good_path)}")

        # Reproject with validation
        reproj_path = tmp / "reproj.tif"
        dm._reproject_with_validation(good_path, reproj_path, "EPSG:32632")
        print(f"Reprojection succeeded: {reproj_path.exists()}")
        print(f"Identity transform after reproject: {dm._has_identity_transform(reproj_path)}")

except ImportError:
    print("rasterio not installed — pip install 'PyGeoFetch[geo]'")

Identity transform detected: True
  (pixel=1.0m, origin=(0,0), projected CRS — geographic coordinates wrong)
Normal WGS84 file detected: False
Reprojection succeeded: True
Identity transform after reproject: False


/home/mrtenkorang/PyGeoFetch-1.0 (2).0-complete/pygeofetch/venv/lib/python3.12/site-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(


## 4. Band Selection

In [19]:
# Sentinel-2 band reference
bands_ref = {
    "B02": ("Blue",            "10m", "RGB"),
    "B03": ("Green",           "10m", "RGB"),
    "B04": ("Red",             "10m", "RGB + NDVI"),
    "B08": ("NIR",             "10m", "NDVI, EVI, NDWI"),
    "B11": ("SWIR-1",          "20m", "Fire, burn, moisture"),
    "B12": ("SWIR-2",          "20m", "NBR, dNBR"),
    "SCL": ("Scene Class",     "20m", "Cloud mask"),
}
print(f"{'Band':<6} {'Name':<20} {'Res':<6} {'Use for'}")
print("-" * 55)
for band, (name, res, use) in bands_ref.items():
    print(f"{band:<6} {name:<20} {res:<6} {use}")

print()
sizes = [
    ("B02,B03,B04",          "RGB",          "~150 MB"),
    ("B04,B08",              "NDVI",         "~100 MB"),
    ("B02,B03,B04,B08",      "RGB+NIR",      "~200 MB"),
    ("B04,B08,B11,B12",      "NDVI+NBR",     "~200 MB"),
    ("(all bands)",          "Full scene",   "~600 MB"),
]
print("Download size by band selection:")
for bands, use, size in sizes:
    print(f"  --bands {bands:<25}  {use:<12}  {size}")

Band   Name                 Res    Use for
-------------------------------------------------------
B02    Blue                 10m    RGB
B03    Green                10m    RGB
B04    Red                  10m    RGB + NDVI
B08    NIR                  10m    NDVI, EVI, NDWI
B11    SWIR-1               20m    Fire, burn, moisture
B12    SWIR-2               20m    NBR, dNBR
SCL    Scene Class          20m    Cloud mask

Download size by band selection:
  --bands B02,B03,B04                RGB           ~150 MB
  --bands B04,B08                    NDVI          ~100 MB
  --bands B02,B03,B04,B08            RGB+NIR       ~200 MB
  --bands B04,B08,B11,B12            NDVI+NBR      ~200 MB
  --bands (all bands)                Full scene    ~600 MB


## 5. Download with Post-Processing Chain

In [20]:
# 9 available post-process actions: unzip, reproject, compress, cog,
# clip, resample, ndvi, pan-sharpen, merge

options = DownloadOptions(
    parallel=2,
    retry_attempts=3,
    resume=True,
    verify_checksum=False,
    on_failure="skip",
    bands=["B02", "B03", "B04", "B08", "B11", "B12"],
    # post_process=[
    #     PostProcessAction.from_string("reproject:EPSG:4326"),
    #     PostProcessAction.from_string("compress:lzw"),
    #     PostProcessAction.from_string("cog"),
    #     # PostProcessAction.from_string("clip:EPSG:4326"),
    #     PostProcessAction.from_string("resample:20m"),
    #     PostProcessAction.from_string("ndvi"),
    #     PostProcessAction.from_string("pan-sharpen"),
    #     PostProcessAction.from_string("merge"),
    # ],
)


dl_results = client.download(items, Path("output/downloads/"), options)

print("Post-process chain:")
for a in options.post_process:
    print(f"  {a.action}  params={a.params}")

print()
print("Download log format (spec-compliant):")
print("  Downloading 2 scene(s) to: ./output/downloads/")
print("  [>>>>>>>>>>>>>>>>>>>>]  1/2  S1C_IW_GRDH_...zip   2.1 GB / 2.1 GB  18.4 MB/s")
print("  checkmark S1C_IW_GRDH_1SDV_20260601T053000          2100 MB  17.8s")
print("  post-processing: reproject:EPSG:4326 -> compress:lzw -> cog")
print("  Download complete: 2/2 succeeded  3900 MB total")

22:11:24 INFO [  downloader] Downloading S2B_18TXK_20240531_0_L2A
22:11:24 INFO [  downloader]   ✓ S2B_18TXK_20240531_0_L2A                         870 MB    0.0s
Post-process chain:

Download log format (spec-compliant):
  [>>>>>>>>>>>>>>>>>>>>]  1/2  S1C_IW_GRDH_...zip   2.1 GB / 2.1 GB  18.4 MB/s
  checkmark S1C_IW_GRDH_1SDV_20260601T053000          2100 MB  17.8s
  post-processing: reproject:EPSG:4326 -> compress:lzw -> cog
  Download complete: 2/2 succeeded  3900 MB total


## 6. Inspect Downloaded Files

In [ ]:

from pathlib import Path

BO2_ = Path("/home/mrtenkorang/PyGeoFetch-1.0 (2).0-complete/pygeofetch/notebooks/output/downloads/aws_earth/B02.tif")
BO3_ = Path("/home/mrtenkorang/PyGeoFetch-1.0 (2).0-complete/pygeofetch/notebooks/output/downloads/aws_earth/B02.tif")
BO4_ = Path("/home/mrtenkorang/PyGeoFetch-1.0 (2).0-complete/pygeofetch/notebooks/output/downloads/aws_earth/B04.tif")
BO8_ = Path("/home/mrtenkorang/PyGeoFetch-1.0 (2).0-complete/pygeofetch/notebooks/output/downloads/aws_earth/B08.tif")
BO11_ = Path("/home/mrtenkorang/PyGeoFetch-1.0 (2).0-complete/pygeofetch/notebooks/output/downloads/aws_earth/B11.tif")
BO12_ = Path("/home/mrtenkorang/PyGeoFetch-1.0 (2).0-complete/pygeofetch/notebooks/output/downloads/aws_earth/B12.tif")

output_ = Path("/home/mrtenkorang/PyGeoFetch-1.0 (2).0-complete/pygeofetch/notebooks/output/downloads/aws_earth/stack.tif")


result = client.indices.stack(
                [BO2_, BO3_, BO4_, BO8_, BO11_, BO12_],output=output_)

In [1]:
from pygeofetch.viz import Plotter,MapViewer
from pathlib import Path

path = Path("/home/mrtenkorang/PyGeoFetch-1.0 (2).0-complete/pygeofetch/notebooks/output/downloads/aws_earth/ndvi.tif")
rgb = Path("/home/mrtenkorang/PyGeoFetch-1.0 (2).0-complete/pygeofetch/notebooks/output/downloads/aws_earth/ndvi.tif")

# pl = Plotter()
# pl.plot_raster(output_, title="Sentinel-2 RGB")
# # pl.plot_histogram(output_,title='RGB')
# pl.plot_rgb(red=BO4_, green=BO3_, blue=BO2_, title="Sentinel-2 RGB")


map = MapViewer()
# map.add_raster(path)
map.add_raster(rgb)
map.show()


Map(center=[40.1403355, -73.1769425], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_titl…

In [ ]:
# Check what was downloaded
dl_dir = Path("output/downloads")
if dl_dir.exists():
    all_files = sorted(dl_dir.rglob("*"))
    tifs = [f for f in all_files if f.suffix.lower() in (".tif", ".tiff")]
    print(f"Downloaded {len(all_files)} total files ({len(tifs)} GeoTIFFs):")
    for f in all_files[:20]:
        size = f.stat().st_size / 1024 / 1024
        print(f"  {f.relative_to(dl_dir)}  ({size:.1f} MB)")
else:
    print("No downloads yet — run a download above first")

Downloaded 16 total files (15 GeoTIFFs):
  aws_earth  (0.0 MB)
  aws_earth/AOT.tif  (0.1 MB)
  aws_earth/B01.tif  (3.9 MB)
  aws_earth/B02.tif  (198.3 MB)
  aws_earth/B03.tif  (197.3 MB)
  aws_earth/B04.tif  (196.1 MB)
  aws_earth/B05.tif  (46.3 MB)
  aws_earth/B06.tif  (45.9 MB)
  aws_earth/B07.tif  (46.0 MB)
  aws_earth/B08.tif  (194.6 MB)
  aws_earth/B09.tif  (4.0 MB)
  aws_earth/B11.tif  (42.0 MB)
  aws_earth/B12.tif  (42.1 MB)
  aws_earth/B8A.tif  (45.5 MB)
  aws_earth/SCL.tif  (0.3 MB)
  aws_earth/WVP.tif  (1.6 MB)


In [ ]:
# Inspect a GeoTIFF with rasterio
try:
    import rasterio
    tifs = sorted(Path("output/downloads").rglob("*.tif"))
    if tifs:
        with rasterio.open(tifs[0]) as src:
            print(f"File:       {tifs[0].name}")
            print(f"Driver:     {src.driver}")
            print(f"CRS:        {src.crs}")
            print(f"Dimensions: {src.width} x {src.height} px")
            print(f"Bands:      {src.count}")
            print(f"Pixel size: {src.res[0]:.1f} x {src.res[1]:.1f} m")
            t = src.transform
            print(f"Origin:     ({t.c:.4f}, {t.f:.4f})")
            print(f"Nodata:     {src.nodata}")
    else:
        print("No GeoTIFFs found — run a download first")
except ImportError:
    print("rasterio not installed — pip install 'PyGeoFetch[geo]'")

File:       AOT.tif
Driver:     GTiff
CRS:        EPSG:32618
Dimensions: 1830 x 1830 px
Bands:      1
Pixel size: 60.0 x 60.0 m
Origin:     (600000.0000, 4500000.0000)
Nodata:     0.0


## 7. All 9 Post-Process Actions

In [ ]:
actions = [
    ("unzip",                 "Extract ZIP/TAR archives"),
    ("reproject:EPSG:4326",   "Reproject to WGS84 (or any CRS)"),
    ("compress:lzw",          "Lossless compression (lzw, deflate, zstd)"),
    ("cog",                   "Cloud Optimized GeoTIFF with tiling + overviews"),
    ("clip:area.geojson",     "Clip raster to polygon boundary"),
    ("resample:30",           "Resample to 30m resolution"),
    ("ndvi",                  "Compute NDVI from Red (B04) + NIR (B08) bands"),
    ("pan-sharpen",           "Fuse panchromatic + multispectral"),
    ("merge",                 "Mosaic overlapping scenes into one raster"),
]
print(f"{'Action':<28} {'Description'}")
print("-" * 70)
for action, desc in actions:
    print(f"  {action:<26}  {desc}")

Action                       Description
----------------------------------------------------------------------
  unzip                       Extract ZIP/TAR archives
  reproject:EPSG:4326         Reproject to WGS84 (or any CRS)
  compress:lzw                Lossless compression (lzw, deflate, zstd)
  cog                         Cloud Optimized GeoTIFF with tiling + overviews
  clip:area.geojson           Clip raster to polygon boundary
  resample:30                 Resample to 30m resolution
  ndvi                        Compute NDVI from Red (B04) + NIR (B08) bands
  pan-sharpen                 Fuse panchromatic + multispectral
  merge                       Mosaic overlapping scenes into one raster


---
## Summary
- BUG 2: len(results) always equals len(items), order preserved
- BUG 3: _validate_downloaded_file() reads first raster tile — catches partial downloads
- BUG 4: _has_identity_transform() + _reproject_with_validation() detect/reject bad reprojections
- Band selection reduces download size by 75% (150 MB vs 600 MB)
- 9 post-process actions available inline after download
- Download log now follows spec format with progress bar

**Next:** Notebook 05 — Pipelines & Scheduling